In [1]:
import multiprocessing

In [2]:
import spacy
from spacy.util import compile_infix_regex
import spacy_alignments as tokenizations
from spacy.tokenizer import Tokenizer
from spacy.tokens import Doc, Token

In [3]:
from transformers import RobertaTokenizerFast, RobertaModel

In [4]:
from data import build_spacy_and_token_cache, create_dataloader, loadi2b2
from train import train_model
from model import create_model
# from evaluate import evaluate_closure

In [5]:
def create_custom_tokenizer(nlp, special_tokens):
    # Lấy phần tử tokenizer mặc định của spaCy
    infix_re = compile_infix_regex(nlp.Defaults.infixes)
    
    # Tạo một custom tokenizer từ spaCy mặc định
    custom_tokenizer = Tokenizer(nlp.vocab, infix_finditer=infix_re.finditer)

    # Thêm các token đặc biệt vào bộ tokenizer
    special_case = {token: [{spacy.symbols.ORTH: token}] for token in special_tokens}
    for token in special_tokens:
        custom_tokenizer.add_special_case(token, special_case[token])
    
    return custom_tokenizer

In [6]:
# Khởi tạo tokenizer và model BERT
tokenizer = RobertaTokenizerFast.from_pretrained("FacebookAI/roberta-base")
bert = RobertaModel.from_pretrained("FacebookAI/roberta-base")

# Thêm token đặc biệt nếu chưa có
special_tokens = {"additional_special_tokens": ["[E1]", "[/E1]", "[E2]", "[/E2]"]}
num_added_toks = tokenizer.add_special_tokens(special_tokens)
bert.resize_token_embeddings(len(tokenizer))  # Resize embedding BERT
# logging.info("Info: BERT loaded")

# Tokenizer và DataLoader
spacy_nlp = spacy.load('en_core_web_sm')
spacy_nlp.tokenizer = create_custom_tokenizer(spacy_nlp, special_tokens['additional_special_tokens'])

Some weights of RobertaModel were not initialized from the model checkpoint at FacebookAI/roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [7]:
data_info = loadi2b2()

In [8]:
train_df = data_info['train_df']
valid_df = data_info['valid_df']
test_df = data_info['test_df']
num_classes = data_info['num_classes']
relation_map = data_info['label_mapping']
id2label = data_info['id2label']

type_map = {"UNKNOWN": 0, "OCCURRENCE": 1, "TREATMENT": 2, "TEST": 3, "DURATION": 4, "PROBLEM": 5, "CLINICAL_DEPT": 6, "EVIDENTIAL": 7,
            "DATE": 8, "FREQUENCY": 9, "TIME": 10,
        }
num_types = 11
rule_filename = 'rules_i2b2.txt'

In [9]:
print("Create dataloader:")
batch_size = 128
window_size = 128
MAX_LENGTH = 256
shuffle=False
n_processor = multiprocessing.cpu_count() - 1
print("batch_size: ", batch_size)
print("window_size: ", window_size)
print("MAX_LENGTH: ", MAX_LENGTH)

Create dataloader:
batch_size:  128
window_size:  128
MAX_LENGTH:  256


In [10]:
train_cache_path = '/data/ddao/TRE/pretrained_models/Reasoning/i2b2/cache/train/'
os.makedirs(train_cache_path, exist_ok=True)
train_token_cache_path = train_cache_path + "token_cache_roberta.pt"
train_spacy_jsonl_path = train_cache_path + "spacy_cache_roberta.jsonl"

test_cache_path = '/data/ddao/TRE/pretrained_models/Reasoning/i2b2/cache/test/'
os.makedirs(test_cache_path, exist_ok=True)
test_token_cache_path = test_cache_path + "token_cache_roberta.pt"
test_spacy_jsonl_path = test_cache_path + "spacy_cache_roberta.jsonl"

valid_cache_path = '/data/ddao/TRE/pretrained_models/Reasoning/i2b2/cache/valid/'
os.makedirs(valid_cache_path, exist_ok=True)
valid_token_cache_path = valid_cache_path + "token_cache_roberta.pt"
valid_spacy_jsonl_path = valid_cache_path + "spacy_cache_roberta.jsonl"

In [11]:
# build_spacy_and_token_cache(
#     df=train_df,              # df có cột: text, label_encoded, entity1_start/end/text, entity2_start/end/text, document_id
#     spacy_nlp=spacy_nlp,
#     tokenizer=tokenizer,
#     window_size=window_size,
#     max_length=MAX_LENGTH,
#     jsonl_path=train_spacy_jsonl_path,
#     token_pt=train_token_cache_path,
#     batch_size=batch_size,
#     n_process=n_processor
# )

# build_spacy_and_token_cache(
#     df=test_df,              # df có cột: text, label_encoded, entity1_start/end/text, entity2_start/end/text, document_id
#     spacy_nlp=spacy_nlp,
#     tokenizer=tokenizer,
#     window_size=window_size,
#     max_length=MAX_LENGTH,
#     jsonl_path=test_spacy_jsonl_path,
#     token_pt=test_token_cache_path,
#     batch_size=batch_size,
#     n_process=n_processor
# )

# build_spacy_and_token_cache(
#     df=valid_df,              # df có cột: text, label_encoded, entity1_start/end/text, entity2_start/end/text, document_id
#     spacy_nlp=spacy_nlp,
#     tokenizer=tokenizer,
#     window_size=window_size,
#     max_length=MAX_LENGTH,
#     jsonl_path=valid_spacy_jsonl_path,
#     token_pt=valid_token_cache_path,
#     batch_size=batch_size,
#     n_process=n_processor
# )

In [12]:
train_loader = create_dataloader(train_token_cache_path, train_spacy_jsonl_path, batch_size=batch_size)

val_loader = create_dataloader(valid_token_cache_path, valid_spacy_jsonl_path, batch_size=64)

test_loader = create_dataloader(test_token_cache_path, test_spacy_jsonl_path, batch_size=batch_size)

In [13]:
# ============================
#  MODEL CONFIG
# ============================
enc_layers     = 2
nhead          = 8
p_drop         = 0.1
step_size      = 0.01
lambda_kl      = 0.1
smooth_tau     = 0.1
lambda_entropy = 0.1
max_steps      = 100
# ============================
#  TRAINING CONFIG
# ===========================
epochs   = 40
save_path    = "/data/ddao/TRE/pretrained_models/Reasoning/i2b2/RoBERTav2.pt"
use_amp      = True                # Mixed precision (khuyên dùng với GPU có tensor core)
lr = 0.00001
lambda_distill = 0.1

In [14]:
model = create_model(
    bert,
    num_classes=num_classes,
    num_types=num_types,
    type_map=type_map,
    relation_map=relation_map,
    enc_layers=enc_layers,
    nhead=nhead,
    p_drop=p_drop,
    step_size=step_size,
    lambda_kl=lambda_kl,
    smooth_tau=smooth_tau,
    lambda_entropy=lambda_entropy,
    max_steps=max_steps,
    rule_file_path=rule_filename
)

[UnifiedNeuralReasoningLayer] Hard Rule: R_Trans_BB
[UnifiedNeuralReasoningLayer] Hard Rule: R_Trans_BO
[UnifiedNeuralReasoningLayer] Hard Rule: R_Trans_OB
[UnifiedNeuralReasoningLayer] Hard Rule: R_Trans_OO
[UnifiedNeuralReasoningLayer] Hard Rule: R_Trans_AA
[UnifiedNeuralReasoningLayer] Hard Rule: R_Trans_AO
[UnifiedNeuralReasoningLayer] Hard Rule: R_Trans_OA
[UnifiedNeuralReasoningLayer] Hard Rule: R_Symm_BA
[UnifiedNeuralReasoningLayer] Hard Rule: R_Symm_AB
[UnifiedNeuralReasoningLayer] Hard Rule: R_Symm_OO
[UnifiedNeuralReasoningLayer] init: rules=16, rel=3, types=11, smooth_tau=0.1, lambda_kl=0.1, lambda_entropy=0.1


In [ ]:
train_model(model,
            train_loader,
            val_loader,
            epochs,
            lr,
            lambda_distill,
            save_path,
            resume_from=None,
            use_amp=True)

Using device: cuda


Evaluate: 100%|██████████| 79/79 [00:32<00:00,  2.45it/s]


Epoch 0: Loss=0.6148 (CE=0.6148, KL=-0.0000, Energy=0.0970) | Val F1=0.7976
  Saved best model to /data/ddao/TRE/pretrained_models/Reasoning/i2b2/RoBERTav2.pt (F1: 0.7976)


Evaluate: 100%|██████████| 79/79 [00:32<00:00,  2.46it/s]


Epoch 1: Loss=0.4328 (CE=0.4328, KL=0.0000, Energy=0.1231) | Val F1=0.8596
  Saved best model to /data/ddao/TRE/pretrained_models/Reasoning/i2b2/RoBERTav2.pt (F1: 0.8596)


...

Evaluate: 100%|██████████| 79/79 [00:32<00:00,  2.44it/s]


Epoch 21: Loss=0.0407 (CE=0.0407, KL=0.0000, Energy=0.1841) | Val F1=0.9040


Evaluate: 100%|██████████| 79/79 [00:29<00:00,  2.66it/s]


Epoch 22: Loss=0.0373 (CE=0.0373, KL=0.0000, Energy=0.1845) | Val F1=0.8981


Epoch 23 (Reasoning):   7%|▋         | 15/222 [00:11<02:36,  1.32it/s]

In [14]:
eval_model = create_model(
    bert,
    num_classes=num_classes,
    num_types=num_types,
    type_map=type_map,
    relation_map=relation_map,
    enc_layers=enc_layers,
    nhead=nhead,
    p_drop=p_drop,
    step_size=step_size,
    lambda_kl=lambda_kl,
    smooth_tau=smooth_tau,
    lambda_entropy=lambda_entropy,
    max_steps=max_steps,
    rule_file_path=rule_filename
)


[UnifiedNeuralReasoningLayer] Hard Rule: R_Trans_BB
[UnifiedNeuralReasoningLayer] Hard Rule: R_Trans_BO
[UnifiedNeuralReasoningLayer] Hard Rule: R_Trans_OB
[UnifiedNeuralReasoningLayer] Hard Rule: R_Trans_OO
[UnifiedNeuralReasoningLayer] Hard Rule: R_Trans_AA
[UnifiedNeuralReasoningLayer] Hard Rule: R_Trans_AO
[UnifiedNeuralReasoningLayer] Hard Rule: R_Trans_OA
[UnifiedNeuralReasoningLayer] Hard Rule: R_Symm_BA
[UnifiedNeuralReasoningLayer] Hard Rule: R_Symm_AB
[UnifiedNeuralReasoningLayer] Hard Rule: R_Symm_OO
[UnifiedNeuralReasoningLayer] init: rules=16, rel=3, types=11, smooth_tau=0.1, lambda_kl=0.1, lambda_entropy=0.1


In [15]:
import torch
print("Load pretrained model:")
save_path    = "/data/ddao/TRE/pretrained_models/Reasoning/i2b2/RoBERTav2.pt"
checkpoint   = torch.load(save_path)
eval_model.load_state_dict(checkpoint['model_state_dict'])

Load pretrained model:


<All keys matched successfully>

In [16]:
from evaluate import evaluate_model, evaluate_closure

In [19]:
cm, f1, p, r = evaluate_model(
    test_loader=test_loader,
    model=eval_model
)

Evaluate: 100%|██████████| 216/216 [02:38<00:00,  1.36it/s]


In [20]:
print('CM:\n', cm)
print('f1:', f1, 'p', p, 'r', r)

CM:
 [[ 1922   289   514]
 [  302 13558  1057]
 [  454   612  8822]]
f1: 0.883052390060621 p 0.884000273606895 r 0.8827460951689067


In [21]:
cm, f1, p, r = evaluate_model(
    test_loader=val_loader,
    model=eval_model
)
print('CM:\n', cm)
print('f1:', f1, 'p', p, 'r', r)

Evaluate: 100%|██████████| 79/79 [00:28<00:00,  2.75it/s]

CM:
 [[ 375   29   88]
 [  47 2491  117]
 [  64  100 1684]]
f1: 0.9109815401138806 p 0.911182340608053 r 0.9109109109109109


In [17]:
F1, P, R, all_extras, all_words_spacy = evaluate_closure(
    test_loader=test_loader,
    model=eval_model,
    id2label=id2label       # Dict: {int_label: str_label}
)

Evaluating (closure): 100%|██████████| 216/216 [02:39<00:00,  1.35it/s]


Using closure-based evaluation for I2B2 (document-level).
====== [I2B2 CLOSURE-BASED EVALUATION] ======
Precision:       0.8835
Recall:          0.8855
F1-score:        0.8845
n_gold:          27530
n_pred:          27530
n_gold_closure:  175550
n_pred_closure:  151242
Evaluation finished in 160.27s.


In [18]:
F1, P, R, all_extras, all_words_spacy = evaluate_closure(
    test_loader=val_loader,
    model=eval_model,
    id2label=id2label       # Dict: {int_label: str_label}
)

Evaluating (closure): 100%|██████████| 79/79 [00:28<00:00,  2.74it/s]

Using closure-based evaluation for I2B2 (document-level).
====== [I2B2 CLOSURE-BASED EVALUATION] ======
Precision:       0.9109
Recall:          0.9111
F1-score:        0.9110
n_gold:          4995
n_pred:          4995
n_gold_closure:  11738
n_pred_closure:  11676
Evaluation finished in 28.91s.


In [43]:
topk_Ps = []
topk_Zs = []
for extras, words_spacy in zip(all_extras, all_words_spacy):
    topk_P_batch, topk_Z_batch = TRE.extract_topk_words(extras, words_spacy, topk=5)
    for i in range(len(topk_P_batch)):
        topk_Ps.append(topk_P_batch[i])
        topk_Zs.append(topk_Z_batch[i])

In [31]:
print("P evidence:")
for idx, word, score in topk_Ps[1]:
    print(word, score)

P evidence:


NameError: name 'topk_Ps' is not defined

In [45]:
print("Z evidence:")
for idx, word, score in topk_Zs[1]:
    print(word, score)

Z evidence:
{'i': 0, 'text': 'Admission', 'lemma': 'Admission', 'pos': 'PROPN', 'tag': 'NNP', 'morph': {'Number': 'Sing'}, 'dep': 'compound', 'span': [0, 9]} 0.0
{'i': 1, 'text': 'Date', 'lemma': 'Date', 'pos': 'PROPN', 'tag': 'NNP', 'morph': {'Number': 'Sing'}, 'dep': 'ROOT', 'span': [10, 14]} 0.0
{'i': 2, 'text': ':', 'lemma': ':', 'pos': 'PUNCT', 'tag': ':', 'morph': {}, 'dep': 'punct', 'span': [15, 16]} 0.0
{'i': 3, 'text': '2012', 'lemma': '2012', 'pos': 'NUM', 'tag': 'CD', 'morph': {'NumType': 'Card'}, 'dep': 'nummod', 'span': [17, 21]} 0.0
{'i': 4, 'text': '-', 'lemma': '-', 'pos': 'SYM', 'tag': 'SYM', 'morph': {}, 'dep': 'punct', 'span': [22, 23]} 0.0


In [34]:
APIKEY = "c2d2bcaf-eb2b-40cb-aa05-1a45b90d3bb4"

In [35]:
#This script will return CUI information for a single search term.
#Optional query parameters are commented out below.

import requests
import argparse

def results_list(string):
    uri = "https://uts-ws.nlm.nih.gov"
    content_endpoint = "/rest/search/current"
    full_url = uri+content_endpoint
    page = 0
    
    try:
        while True:
            page += 1
            query = {'string':string,'apiKey':APIKEY, 'pageNumber':page}
            #query['includeObsolete'] = 'true'
            #query['includeSuppressible'] = 'true'
            #query['returnIdType'] = "sourceConcept"
            #query['sabs'] = "SNOMEDCT_US"
            r = requests.get(full_url,params=query)
            r.raise_for_status()
            print(r.url)
            r.encoding = 'utf-8'
            outputs  = r.json()
        
            items = (([outputs['result']])[0])['results']
            
            if len(items) == 0:
                if page == 1:
                    print('No results found.'+'\n')
                    break
                else:
                    break
            
            print("Results for page " + str(page)+"\n")
            
            for result in items:
                print('UI: ' + result['ui'])
                print('URI: ' + result['uri'])
                print('Name: ' + result['name'])
                print('Source Vocabulary: ' + result['rootSource'])
                print('\n')
                
        print('*********')

    except Exception as except_error:
        print(except_error)



In [36]:
results_list('hemilaminectomy')

https://uts-ws.nlm.nih.gov/rest/search/current?string=hemilaminectomy&apiKey=c2d2bcaf-eb2b-40cb-aa05-1a45b90d3bb4&pageNumber=1
Results for page 1

UI: C0442936
URI: https://uts-ws.nlm.nih.gov/rest/content/2025AA/CUI/C0442936
Name: Hemilaminectomy
Source Vocabulary: SNOMEDCT_US


UI: C1735609
URI: https://uts-ws.nlm.nih.gov/rest/content/2025AA/CUI/C1735609
Name: Spinal hemilaminectomy
Source Vocabulary: MDR


UI: C0196273
URI: https://uts-ws.nlm.nih.gov/rest/content/2025AA/CUI/C0196273
Name: Hemilaminectomy for decompression and exploration
Source Vocabulary: SNOMEDCT_US


UI: C2066834
URI: https://uts-ws.nlm.nih.gov/rest/content/2025AA/CUI/C2066834
Name: decompressive hemilaminectomy with partial facetectomy
Source Vocabulary: MEDCIN


UI: C0408611
URI: https://uts-ws.nlm.nih.gov/rest/content/2025AA/CUI/C0408611
Name: Revisional hemilaminectomy excision of cervical intervertebral disc
Source Vocabulary: SNOMEDCT_US


UI: C2711473
URI: https://uts-ws.nlm.nih.gov/rest/content/2025AA/CUI/

In [ ]:
#This script returns UMLS CUIs based on an input file of strings, where each line in txt file is a separate string.
#If no results are found for a specific string, this will be noted in output and output file.
#Each set of results for a string is separated in the output file with '***'.

import requests
import argparse

parser = argparse.ArgumentParser(description='process user given parameters')
parser.add_argument('-k', '--apikey', required = True, dest = 'apikey', help = 'enter api key from your UTS Profile')
parser.add_argument('-v', '--version', required =  False, dest='version', default = 'current', help = 'enter version example-2015AA')
parser.add_argument('-o', '--outputfile', required = True, dest = 'outputfile', help = 'enter a name for your output file')
parser.add_argument('-s', '--sabs', required = False, dest='sabs',help = 'enter a comma-separated list of vocabularies without spaces, like MSH,SNOMEDCT_US,RXNORM')
parser.add_argument('-t', '--ttys', required = False, dest='ttys',help = 'enter a comma-separated list of term types, like PT,SY,IN')
parser.add_argument('-i', '--inputfile', required = True, dest = 'inputfile', help = 'enter a name for your input file')

args = parser.parse_args()
apikey = args.apikey
version = args.version
outputfile = args.outputfile
inputfile = args.inputfile
sabs = args.sabs
ttys = args.ttys

base_uri = 'https://uts-ws.nlm.nih.gov'
string_list = []

with open(inputfile, encoding='utf-8') as f:
    for line in f:
        if line.isspace() is False:    
            strings = line.strip()
            string_list.append(strings)
        else:
            continue

print(string_list)

with open(outputfile, 'w', encoding='utf-8') as o:
    for string in string_list:
        page = 0
        
        o.write('SEARCH STRING: ' + string + '\n' + '\n')
        
        while True:
            page += 1
            path = '/search/'+version
            query = {'string':string, 'apiKey':apikey, 'rootSource':sabs, 'termType':ttys, 'pageNumber':page}
            output = requests.get(base_uri+path, params=query)
            output.encoding = 'utf-8'
            #print(output.url)
        
            outputJson = output.json()
            results = (([outputJson['result']])[0])['results']
            
            if len(results) == 0:
                if page == 1:
                    print('No results found for ' + string +'\n')
                    o.write('No results found.' + '\n' + '\n')
                    break
                else:
                    break
            
            for item in results:
                o.write('UI: ' + item['ui'] + '\n' + 'Name: ' + item['name'] + '\n'  + 'URI: ' + item['uri'] + '\n' + 'Source Vocabulary: ' + item['rootSource'] + '\n' + '\n')
        
        o.write('***' + '\n' + '\n')